#### Random State
- Setting a random_state ensures that the results are reproducible. This is particularly important when sharing your work with others or when you need to debug your code. By using the same random_state, you can ensure that others can replicate your results exactly.

- Consistency in Model Evaluation: When comparing different models or tuning hyperparameters, it's crucial to have a consistent train-test split. Using the same random_state ensures that the evaluation metrics are comparable across different runs.

- During the development phase, you might need to debug your code or test different configurations. A fixed random_state helps in maintaining consistency, making it easier to identify issues and test changes.


In [ ]:
RS=42

## Import NumPY and Pandas

Import the pandas and numpy libraries used for data manipulation and numerical operations.

In [ ]:
import pandas as pd
import numpy as np

## Load Iris Dataset w/scikit-learn

Imported `load_iris` function from `sklearn.datasets`, then assign to `irish_bunch` variable. At this point `iris_bunch` is `Bunch` type variable. Converting `Bunch` type of variable to `pd.DataFrame` helps us to modify data and work with `sklearn`.

---

`Bunch` variable kind a child of `dict`, so it has keys and values. When attaching `Bunch` variable to `pd.Dataframe` used `data` key as data, and `feature_names` key as column names, then concated `target` key as `class`.

In [ ]:
from sklearn.datasets import load_iris
iris_bunch = load_iris()
print(iris_bunch.DESCR)

Create a pandas DataFrame from the loaded Iris data, using feature names as columns and adding the target labels as a 'class' column, then preview the first rows.


In [ ]:
df = pd.DataFrame(data=iris_bunch.data, columns=iris_bunch.feature_names)
df['class'] = iris_bunch.target
df.head()

### Data Inspectation

`df.info()` prints general information about dataset. Good news, there is no `null` values in iris dataset, and all metrics are in `centimeter (cm)`.

In [ ]:
df.info()

## Model Training

`KFold` splits data to k folds, then train a model with each fold. `cross_val_score` scores each model from folds.

`KNeighborsClassifier` is class to create `kNN` object.  `n_neighbors` paramteres refers `k` in `kNN`.

Code below iterates 1, 3, 5, 7 `k` neighbours nested with 3 and 5 `f` folds. Each iteration appends general report that includes `k_fold`, `k_neighbours`,`scores`(each score of fold),`mean_accuracy`

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

X = df.drop(columns=['class'])
y = df['class']
reports = []
models = []

for k_fold in [3, 5]:
  for k_neighbors in [1, 3, 5, 7]:
    kf = KFold(n_splits=k_fold, random_state=RS, shuffle=True)

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k_neighbors))
    ])


    models.append(pipeline)

    scores = cross_val_score(pipeline, X, y, cv=kf)

    reports.append({
        'k_fold': k_fold,
        'k_neighbors': k_neighbors,
        'scores': scores,
        'mean_accuracy': scores.mean()
    })

Plot the mean cross-validation accuracy of k-NN against the number of neighbors for each fold configuration.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

reports_df = pd.DataFrame(reports)

plt.figure(figsize=(10, 5))

for fold in reports_df['k_fold'].unique():
    subset = reports_df[reports_df['k_fold'] == fold]
    plt.plot(subset['k_neighbors'], subset['mean_accuracy'], marker='o', label=f'{fold}-Fold CV')

plt.title('k-NN Accuracy vs. Number of Neighbors (k)', fontsize=14)
plt.xlabel('Number of Neighbors (k)', fontsize=12)
plt.ylabel('Mean Accuracy', fontsize=12)
plt.xticks(reports_df['k_neighbors'].unique())
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title='Validation Strategy')
plt.tight_layout()

plt.show()

Display a bar chart comparing the mean cross-validation accuracy of all trained k-NN model configurations.


In [ ]:
import matplotlib.pyplot as plt

labels = [f"{r['k_fold']}-Fold (k={r['k_neighbors']})" for r in reports]
mean_accuracies = [r['mean_accuracy'] for r in reports]

plt.figure(figsize=(10, 5))
bars = plt.bar(labels, mean_accuracies, color='skyblue', edgecolor='black')

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.005, f"{yval:.4f}", ha='center', va='bottom', fontsize=10)

plt.title('Mean Cross-Validation Accuracy of the Trained Models', fontsize=14)
plt.xlabel('Model Configuration (Fold and Neighbors)', fontsize=12)
plt.ylabel('Mean Accuracy', fontsize=12)
plt.ylim(min(mean_accuracies) - 0.02, max(mean_accuracies) + 0.02)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

Fit each trained k-NN model on the two most important features and visualize their decision boundaries alongside the Iris data points.


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import math
from sklearn.inspection import DecisionBoundaryDisplay

X_2d = X[['petal length (cm)', 'petal width (cm)']]

num_models = len(models)
cols = 4
rows = math.ceil(num_models / cols)

fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows), sharex=True, sharey=True)
axes = axes.flatten()

cmap_colors = mpl.colormaps["Set1"]

for idx, model in enumerate(models):
    ax = axes[idx]

    from sklearn.base import clone
    knn_2d = clone(model)
    knn_2d.fit(X_2d, y)

    DecisionBoundaryDisplay.from_estimator(
        knn_2d,
        X_2d,
        response_method="predict",
        cmap=cmap_colors,
        plot_method="pcolormesh",
        shading="auto",
        ax=ax,
        xlabel="Petal Length (cm)",
        ylabel="Petal Width (cm)",
    )

    for class_idx, class_name in enumerate(iris_bunch.target_names):
        mask = (y == class_idx)
        ax.scatter(
            X_2d.loc[mask, 'petal length (cm)'],
            X_2d.loc[mask, 'petal width (cm)'],
            color=cmap_colors(class_idx/2),
            edgecolors='k',
            label=class_name if idx == 0 else "", # only label first plot for clear legend
            s=30
        )

    report = reports[idx]
    ax.set_title(f"{report['k_fold']}-Fold (k={report['k_neighbors']})\nAcc: {report['mean_accuracy']:.4f}")

for idx in range(num_models, len(axes)):
    fig.delaxes(axes[idx])

axes[0].legend(title="Iris Classes", loc="upper left")
plt.tight_layout()
plt.show()

## Predicting w/Naive Bayes

Train a Gaussian Naive Bayes classifier with cross-validation and evaluate it using a confusion matrix, ROC curves, and precision-recall curves.

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc, precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

X = df.drop(columns=['class'])
y = df['class']
n_classes = len(np.unique(y))

kf = KFold(n_splits=5, random_state=RS, shuffle=True)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('nb', GaussianNB())
])

y_pred = cross_val_predict(pipeline, X, y, cv=kf, method='predict') # Class predictions 0/1/2
y_score = cross_val_predict(pipeline, X, y, cv=kf, method='predict_proba') # Scores w/gausian naive bayes

y_binarized = label_binarize(y, classes=np.unique(y))

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

cm = confusion_matrix(y, y_pred)
cm_display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=iris_bunch.target_names)
cm_display.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Confusion Matrix (OOF)')

for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_binarized[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    axes[1].plot(fpr, tpr, label=f'{iris_bunch.target_names[i]} (AUC = {roc_auc:.2f})')

axes[1].plot([0, 1], [0, 1], 'k--', label='Random Guess')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves (One-vs-Rest)')
axes[1].legend(loc='lower right')
axes[1].grid(True, linestyle='--', alpha=0.5)

for i in range(n_classes):
    precision, recall, _ = precision_recall_curve(y_binarized[:, i], y_score[:, i])
    avg_precision = average_precision_score(y_binarized[:, i], y_score[:, i])
    axes[2].plot(recall, precision, label=f'{iris_bunch.target_names[i]} (AP = {avg_precision:.2f})')

axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curves (One-vs-Rest)')
axes[2].legend(loc='lower left')
axes[2].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Reference
- https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html
- https://scikit-learn.org/stable/auto_examples/neighbors/plot_classification.html
- https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html
- https://matplotlib.org/stable/users/explain/colors/colormaps.html#cyclic
- https://www.geeksforgeeks.org/machine-learning/what-is-scikit-learn-random-state-in-splitting-dataset/#what-israndom_state
- Google Colab integrated Gemini for data visualizations and plotting.
- https://scikit-learn.org/stable/modules/naive_bayes.html#gaussian-naive-bayes

###